In [1]:
import torch
import torchvision
import torch.nn as nn
from torchvision import transforms , datasets
from torch.utils.data import DataLoader , TensorDataset

##Uses of the Torchvision.transforms

computer vision modlue and uses for
1. Preprocessing
2. Data Augmentation
3. format Conversion


**Data Preprocessing & Standardization:** Neural networks require fixed input dimensions and well-scaled numerical ranges. Transforms resize, crop, and normalize pixel intensities (such as matching ImageNet channel means and standard deviations) so every batch has consistent shape and scale.

**Data Augmentation:** During training, random spatial and color perturbations (flips, rotations, crops, color jitter) generate varied views of existing images on the fly, reducing overfitting without collecting extra data.

  **Format & Type Conversion:** Converts raw PIL.Image objects or NumPy arrays into PyTorch tensors with channel-first layout (C, H, W) and converts pixel types from uint8 ([0, 255]) to scaled float32 ([0.0, 1.0]).
  
  
  **Pipeline Composition:** Chains multiple operations sequentially using Compose so an entire preprocessing workflow can be passed directly into a PyTorch Dataset.


In [2]:
image_path = './'
transform = transforms.Compose([transforms.ToTensor()])   #convert an image or a numPy array into the Pytorch Tensors

mnist_train_dataset = torchvision.datasets.MNIST(root = image_path, train  =True , transform = transform , download =True)

mnist_test_dataset = torchvision.datasets.MNIST(root = image_path , train=False , transform = transform , download=True)

batch_size = 64
torch.manual_seed(1)
train_dl  = DataLoader(mnist_train_dataset , batch_size , shuffle=True)
test_dl = DataLoader(mnist_test_dataset , batch_size , shuffle=False)



100%|██████████| 9.91M/9.91M [00:00<00:00, 15.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 412kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.68MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.67MB/s]


In [3]:
mnist_train_dataset.data.shape # this is to chack the whole data

torch.Size([60000, 28, 28])

In [4]:
hidden_units = [32, 16]

image_size = mnist_train_dataset[0][0].shape
input_size = image_size[0] * image_size[1] * image_size[2]
all_layers = [nn.Flatten()] #reshapes a contiguous range of tensor dimensions into a single dimension

for i in hidden_units:
  layer = nn.Linear(input_size , i)
  all_layers.append(layer)
  all_layers.append(nn.ReLU())
  input_size = i

all_layers.append(nn.Linear(hidden_units[-1] , 10))
model = nn.Sequential(*all_layers)
model

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=32, bias=True)
  (2): ReLU()
  (3): Linear(in_features=32, out_features=16, bias=True)
  (4): ReLU()
  (5): Linear(in_features=16, out_features=10, bias=True)
)

This model starts with the flatten layer that flatten input image into 1 d tensor
uses of the

nn.Flatten() -> Collapses contiguous axes into one  [64, 1, 28, 28] → [64, 784]

squeeze(dim) -> Removes dimensions whose size equals[64, 1, 28, 28] → [64, 28, 28]

unsqueeze(dim) -> Inserts a new dimension of size 1 at dim [28, 28] → [1, 28, 28]

view(*shape)	Re-interprets tensor into any compatible shape (requires contiguous memory)	[64, 1, 28, 28] → [64, 784]	Fast zero-copy reshaping when memory is contiguous


reshape(*shape)	Same as view(), but safely copies memory if tensor is non-contiguous	[64, 1, 28, 28] → [64, 4, 196]	General-purpose reshaping without contiguity errors

In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters() , lr=0.001)
torch.manual_seed(1)
num_epochs = 20

for epoch in range(num_epochs):
  accuracy_hist_train = 0
  for x_batch , y_batch in train_dl:
    pred = model(x_batch)
    loss = loss_fn(pred , y_batch)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    is_correct  = (torch.argmax(pred , dim=1)==y_batch).float()
    accuracy_hist_train += is_correct.sum()
  accuracy_hist_train /= len(train_dl.dataset)
  print(f'epoch : {epoch} || Accuracy : {accuracy_hist_train:.4f}')

epoch : 0 || Accuracy : 0.1057
epoch : 1 || Accuracy : 0.2183
epoch : 2 || Accuracy : 0.2912
epoch : 3 || Accuracy : 0.3513
epoch : 4 || Accuracy : 0.3952
epoch : 5 || Accuracy : 0.4781
epoch : 6 || Accuracy : 0.5404
epoch : 7 || Accuracy : 0.5965
epoch : 8 || Accuracy : 0.6761
epoch : 9 || Accuracy : 0.7562
epoch : 10 || Accuracy : 0.7928
epoch : 11 || Accuracy : 0.8101
epoch : 12 || Accuracy : 0.8210
epoch : 13 || Accuracy : 0.8304
epoch : 14 || Accuracy : 0.8377
epoch : 15 || Accuracy : 0.8438
epoch : 16 || Accuracy : 0.8490
epoch : 17 || Accuracy : 0.8535
epoch : 18 || Accuracy : 0.8579
epoch : 19 || Accuracy : 0.8613
